In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

In [25]:
df = pd.read_excel("Online Retail.xlsx")

In [26]:
# Menampilkan 5 baris pertama
display(df.head())

# Menampilkan jumlah baris dan kolom
print("Baris:", df.shape[0])
print("Kolom:", df.shape[1])

# Menampilkan informasi struktur dataset
df.info()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


Baris: 541909
Kolom: 8
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


In [27]:
# Memilih kolom yang relevan untuk Association Rule
df_ar = df[['InvoiceNo', 'Description']].copy()
df_ar.head()

,InvoiceNo,Description
0,536365,WHITE HANGING HEART T-LIGHT HOLDER
1,536365,WHITE METAL LANTERN
2,536365,CREAM CUPID HEARTS COAT HANGER
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE
4,536365,RED WOOLLY HOTTIE WHITE HEART.


In [28]:
df_ar = df_ar.dropna(subset=['Description'])
df_ar['InvoiceNo'] = df_ar['InvoiceNo'].astype(str)
df_ar = df_ar[~df_ar['InvoiceNo'].str.startswith('C')]

In [29]:
df_ar = df_ar[df.loc[df_ar.index, 'Quantity'] > 0]

In [30]:
print("Sebelum pembersihan:")
print("Jumlah baris   :", df.shape[0])
print("Jumlah invoice :", df['InvoiceNo'].nunique())

print("\nSesudah pembersihan:")
print("Jumlah baris   :", df_ar.shape[0])
print("Jumlah invoice :", df_ar['InvoiceNo'].nunique())

Sebelum pembersihan:
Jumlah baris   : 541909
Jumlah invoice : 25900

Sesudah pembersihan:
Jumlah baris   : 530693
Jumlah invoice : 20136


In [31]:
transactions = df_ar.groupby('InvoiceNo')['Description'].apply(list).tolist()

In [32]:
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
basket = pd.DataFrame(te_array, columns=te.columns_)

basket.head()

,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,I LOVE LONDON MINI RUCKSACK,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,...,returned,taig adjust,test,to push order througha s stock was,website fixed,wrongly coded 20713,wrongly coded 23343,wrongly marked,wrongly marked 23343,wrongly sold (22719) barcode
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [33]:
frequent_itemsets = fpgrowth(basket, min_support=0.02, use_colnames=True)
print("Jumlah frequent itemset:", len(frequent_itemsets))
display(frequent_itemsets.sort_values('support', ascending=False).head(10))

rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)
print("Jumlah rules:", len(rules))
display(rules.sort_values('lift', ascending=False)[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

Jumlah frequent itemset: 375


,support,itemsets
0,0.112237,frozenset({WHITE HANGING HEART T-LIGHT HOLDER})
66,0.103894,frozenset({JUMBO BAG RED RETROSPOT})
162,0.098778,frozenset({REGENCY CAKESTAND 3 TIER})
232,0.083731,frozenset({PARTY BUNTING})
29,0.077672,frozenset({LUNCH BAG RED RETROSPOT})
4,0.072259,frozenset({ASSORTED COLOUR BIRD ORNAMENT})
243,0.068782,frozenset({SET OF 3 CAKE TINS PANTRY DESIGN })
30,0.065554,frozenset({PACK OF 72 RETROSPOT CAKE CASES})
107,0.063220,frozenset({LUNCH BAG BLACK SKULL.})
56,0.062028,frozenset({NATURAL SLATE HEART CHALKBOARD })


Jumlah rules: 176


,antecedents,consequents,support,confidence,lift
139,"frozenset({ROSES REGENCY TEACUP AND SAUCER , GREEN REGENCY TEACUP AND SAUCER})",frozenset({PINK REGENCY TEACUP AND SAUCER}),0.026917,0.705729,18.551648
142,frozenset({PINK REGENCY TEACUP AND SAUCER}),"frozenset({ROSES REGENCY TEACUP AND SAUCER , GREEN REGENCY TEACUP AND SAUCER})",0.026917,0.707572,18.551648
143,frozenset({GREEN REGENCY TEACUP AND SAUCER}),"frozenset({ROSES REGENCY TEACUP AND SAUCER , PINK REGENCY TEACUP AND SAUCER})",0.026917,0.533990,17.950627
138,"frozenset({ROSES REGENCY TEACUP AND SAUCER , PINK REGENCY TEACUP AND SAUCER})",frozenset({GREEN REGENCY TEACUP AND SAUCER}),0.026917,0.904841,17.950627
135,frozenset({GREEN REGENCY TEACUP AND SAUCER}),frozenset({PINK REGENCY TEACUP AND SAUCER}),0.031436,0.623645,16.393893
134,frozenset({PINK REGENCY TEACUP AND SAUCER}),frozenset({GREEN REGENCY TEACUP AND SAUCER}),0.031436,0.826371,16.393893
141,frozenset({ROSES REGENCY TEACUP AND SAUCER }),"frozenset({PINK REGENCY TEACUP AND SAUCER, GREEN REGENCY TEACUP AND SAUCER})",0.026917,0.508443,16.173782
140,"frozenset({PINK REGENCY TEACUP AND SAUCER, GREEN REGENCY TEACUP AND SAUCER})",frozenset({ROSES REGENCY TEACUP AND SAUCER }),0.026917,0.856240,16.173782
175,frozenset({GARDENERS KNEELING PAD KEEP CALM }),frozenset({GARDENERS KNEELING PAD CUP OF TEA }),0.027116,0.598028,15.886413
174,frozenset({GARDENERS KNEELING PAD CUP OF TEA }),frozenset({GARDENERS KNEELING PAD KEEP CALM }),0.027116,0.720317,15.886413


In [34]:
frequent_itemsets = fpgrowth(basket, min_support=0.01, use_colnames=True)

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)
rules = rules[rules['lift'] > 1]

print("Jumlah frequent itemset:", len(frequent_itemsets))
print("Jumlah rules:", len(rules))
# Rules yang dihasilkan terlalu banyak (841), sehingga sulit dibaca dan diinterpretasi satu per satu.

Jumlah frequent itemset: 1854
Jumlah rules: 841


In [35]:
hasil = []
for s, c in [(0.01, 0.5), (0.015, 0.5), (0.02, 0.5), (0.015, 0.6), (0.02, 0.6)]:
    fi = fpgrowth(basket, min_support=s, use_colnames=True)
    r = association_rules(fi, metric="confidence", min_threshold=c)
    r = r[r['lift'] > 1]
    hasil.append({
        'min_support': s,
        'min_confidence': c,
        'frequent_itemset': len(fi),
        'rules': len(r)
    })

pd.DataFrame(hasil)
# Jumlah rules (61) cukup untuk dianalisis dan tidak terlalu banyak seperti 0,01 maupun 0,015 (841 dan 177)
# Tidak terlalu sedikit seperti 31 rules pada confidence 0,6, yang bisa membuang pola yang masih bermakna

,min_support,min_confidence,frequent_itemset,rules
0,0.010,0.5,1854,841
1,0.015,0.5,694,177
2,0.020,0.5,375,61
3,0.015,0.6,694,94
4,0.020,0.6,375,31


In [36]:
frequent_itemsets = fpgrowth(basket, min_support=0.02, use_colnames=True)

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)
rules = rules[rules['lift'] > 1]

print("Jumlah frequent itemset:", len(frequent_itemsets))
print("Jumlah rules:", len(rules))

Jumlah frequent itemset: 375
Jumlah rules: 61


In [37]:
rules_tampil = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].copy()

rules_tampil['antecedents'] = rules_tampil['antecedents'].apply(lambda x: ', '.join(x))
rules_tampil['consequents'] = rules_tampil['consequents'].apply(lambda x: ', '.join(x))

rules_tampil = rules_tampil.sort_values('lift', ascending=False).reset_index(drop=True)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 250)

rules_tampil

,antecedents,consequents,support,confidence,lift
0,PINK REGENCY TEACUP AND SAUCER,"ROSES REGENCY TEACUP AND SAUCER , GREEN REGENCY TEACUP AND SAUCER",0.026917,0.707572,18.551648
1,"ROSES REGENCY TEACUP AND SAUCER , GREEN REGENCY TEACUP AND SAUCER",PINK REGENCY TEACUP AND SAUCER,0.026917,0.705729,18.551648
2,"ROSES REGENCY TEACUP AND SAUCER , PINK REGENCY TEACUP AND SAUCER",GREEN REGENCY TEACUP AND SAUCER,0.026917,0.904841,17.950627
3,GREEN REGENCY TEACUP AND SAUCER,"ROSES REGENCY TEACUP AND SAUCER , PINK REGENCY TEACUP AND SAUCER",0.026917,0.533990,17.950627
4,GREEN REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER,0.031436,0.623645,16.393893
5,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,0.031436,0.826371,16.393893
6,ROSES REGENCY TEACUP AND SAUCER,"PINK REGENCY TEACUP AND SAUCER, GREEN REGENCY TEACUP AND SAUCER",0.026917,0.508443,16.173782
7,"PINK REGENCY TEACUP AND SAUCER, GREEN REGENCY TEACUP AND SAUCER",ROSES REGENCY TEACUP AND SAUCER,0.026917,0.856240,16.173782
8,GARDENERS KNEELING PAD KEEP CALM,GARDENERS KNEELING PAD CUP OF TEA,0.027116,0.598028,15.886413
9,GARDENERS KNEELING PAD CUP OF TEA,GARDENERS KNEELING PAD KEEP CALM,0.027116,0.720317,15.886413


In [38]:
top3 = rules_tampil.head(3)
top3

,antecedents,consequents,support,confidence,lift
0,PINK REGENCY TEACUP AND SAUCER,"ROSES REGENCY TEACUP AND SAUCER , GREEN REGENCY TEACUP AND SAUCER",0.026917,0.707572,18.551648
1,"ROSES REGENCY TEACUP AND SAUCER , GREEN REGENCY TEACUP AND SAUCER",PINK REGENCY TEACUP AND SAUCER,0.026917,0.705729,18.551648
2,"ROSES REGENCY TEACUP AND SAUCER , PINK REGENCY TEACUP AND SAUCER",GREEN REGENCY TEACUP AND SAUCER,0.026917,0.904841,17.950627


Interpretasi bisnis

Pink -> Green + Roses. Sekitar 2,7% dari seluruh transaksi (sekitar 540 invoice) memuat ketiga cangkir sekaligus. Dari pelanggan yang membeli Pink, 70,8% juga membeli Green dan Roses. Lift 18,55 berarti peluang ketiganya dibeli bersama sekitar 18 kali lebih tinggi dibanding jika produk-produk itu dibeli secara independen. Pelanggan yang tertarik pada satu warna cenderung melengkapi koleksinya dengan dua warna lainnya.

Green + Roses -> Pink. Arah kebalikannya hampir sama kuat: 70,6% pelanggan yang sudah mengambil Green dan Roses juga membeli Pink. Ini menunjukkan ketiga warna dibeli sebagai satu set, bukan karena salah satu menjadi pemicu.

Green -> Pink + Roses. Confidence-nya paling rendah (53,4%) tetapi masih di atas separuh. Pembeli Green lebih sering membeli satu atau dua warna lain daripada langsung ketiganya, sehingga rule ini lebih lemah sebagai dasar rekomendasi.